# Encoder Whisper (beku) + Logistic Regression
        **audio › speaker_id** · notebook `whisper_encoder_logreg`

        **Kapan dipakai:** Embedding dari encoder Whisper-tiny (384 dim) yang dilatih multibahasa pada audio sangat banyak; sering sangat kuat untuk kata/bahasa/pembicara.

        **Jebakan umum:** Whisper memproses jendela 30 detik: klip pendek di-pad, hanya frame awal yang dirata-rata (sudah ditangani).

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Identifikasi 10 pembicara (150 latih / 60 uji, kata campuran; data kecil dan sulit). Bobot pretrained dari `pretrained/` (offline); GPU otomatis bila ada.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Format data: folder WAV + CSV `path,label` (path relatif ke `DATA_ROOT`). Klip dipotong/di-pad ke `LENGTH` sampel (16 kHz).

In [ ]:
DATA_ROOT = ROOT / "data_latihan/audio/speaker/speech_commands_speakers"
TRAIN_CSV, TEST_CSV = DATA_ROOT / "train.csv", DATA_ROOT / "test.csv"
TEST_LABELS = DATA_ROOT / "test_labels.csv"        # kunci jawaban test (hanya latihan); None di lomba
LENGTH = 16000                                        # 1 detik
SEED, VALID_SIZE = 42, 0.2
MODEL_KEY = "whisper_encoder_logreg"
OUT = out_dir("audio", "speaker_id", MODEL_KEY); set_seed(SEED)

## 2. Baca audio

In [ ]:
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.audio import load_wavs, plot_wave_mel
tr, te = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
le = LabelEncoder(); y = le.fit_transform(tr["label"]); CLASSES = list(le.classes_)
X = load_wavs(tr["path"], DATA_ROOT, LENGTH); Xte = load_wavs(te["path"], DATA_ROOT, LENGTH)
print(X.shape, Xte.shape, "| kelas:", CLASSES, "| durasi:", LENGTH / 16000, "dtk")
plot_wave_mel(X[0], f"contoh '{tr['label'][0]}'"); plt.show()
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 3. Embedding pretrained (beku, di-cache)
Bobot beku => aman dihitung sebelum split.

In [ ]:
from superdb.audio import WhisperEmbedder, mfcc_stats
import time; t0 = time.time(); emb = WhisperEmbedder()
E, Ete = emb.encode(X), emb.encode(Xte); print(f"embedding {E.shape} dalam {time.time() - t0:.0f} dtk")
Etr, Eva, ytr_, yva_ = train_test_split(E, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 4. Classifier linear di atas embedding

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))

## 5. CV, evaluasi, dan pembanding MFCC (data sama)

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = cross_val_score(model, Etr, ytr_, cv=StratifiedKFold(3, shuffle=True, random_state=SEED), scoring="f1_macro"); print(f"CV f1_macro = {cv.mean():.4f} ± {cv.std():.4f}")
model.fit(Etr, ytr_); pred = model.predict(Eva); proba = model.predict_proba(Eva)
metrics = classification_metrics(yva_, pred, proba, classes=np.arange(len(CLASSES))); display(show_table(metrics))
from sklearn.linear_model import LogisticRegression
base = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000)).fit(mfcc_stats(Xtr), ytr)
f1b = float((base.predict(mfcc_stats(Xva)) == yva).mean()); print(f"akurasi validation: embedding {metrics['accuracy']:.3f} vs MFCC+LogReg {f1b:.3f}"); metrics["acc_mfcc_baseline"] = f1b
plot_confusion(yva_, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 6. Latih ulang pada semua data → submission → simpan

In [ ]:
from sklearn.base import clone
final = clone(model).fit(E, y); PRED_TE = final.predict(Ete)

In [ ]:
sub = pd.DataFrame({"id": te["id"], "label": [CLASSES[i] for i in PRED_TE]}); sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
if TEST_LABELS is not None and Path(TEST_LABELS).exists():
    truth = pd.read_csv(TEST_LABELS); acc = float((truth.set_index("id").loc[sub["id"], "label"].to_numpy() == sub["label"].to_numpy()).mean())
    print(f"akurasi pada test (kunci jawaban): {acc:.4f}"); metrics["test_accuracy"] = acc

In [ ]:
save_model({"clf": final, "classes": CLASSES}, OUT / "model.joblib"); save_json({"cv_f1_macro": float(cv.mean()), "valid": metrics}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib"); assert (b["clf"].predict(Ete[:30]) == final.predict(Ete[:30])).all(); print("OK. hasil di", OUT)

## Langkah berikutnya
- Gabungkan embedding wav2vec2 + Whisper + MFCC (hstack) lalu classifier.
- Fine-tuning seluruh model biasanya naik lagi, tetapi mahal; linear probe sering sudah cukup.